# Lab 2: Compare Commute Costs with Pandas

**COMPSS 211A | Fall 2026 | Student copy**

**Friday, September 18, 2026**

Use Monday's filtering, functions, derived columns, and `groupby` ideas to analyze synthetic commute records. You will build and inspect results in Python.

## Today’s rhythm

- **0–10 minutes:** Review Quiz 2.
- **10–20 minutes:** Type along with a slow, five-line warm-up. Predict each result before running the line.
- **20–50 minutes:** Complete four duo coding tasks. One person types and one checks the result; switch roles after Task 2. The instructor and GSI will check in as you work.
- **50–60 minutes:** Take Quiz 3.

This is a no-AI practice session. Use Monday's notebook, pandas documentation, your partner, and the teaching team. Run your code and inspect its output; no written responses are needed.

## Analysis question

In these **synthetic** records, which academic program has the highest median monthly transportation cost among respondents who commute to campus? How many known costs are behind that median? Does the program ordering change if you use the mean instead?

The data are made up, so your result is practice with pandas, not a claim about actual Berkeley students. The four tasks below produce the tables needed to answer the question.

## Supplied setup

Run this cell once. It loads the synthetic commute table and checks the course environment. We will not unpack the setup code today.

In [48]:
import platform
import sys
from pathlib import Path
import pandas as pd
import numpy as np

data_url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/9f4e28652fe776220d8470f03b681efddaf58192/data/hw1_after_dark_commute_survey.csv"
commute = pd.read_csv(data_url)

print("Python version:", platform.python_version())
print("Python 3.13:", platform.python_version().startswith("3.13."))
print("pandas version:", pd.__version__)
print("pandas matches 2.2.3:", pd.__version__ == "2.2.3")
print("NumPy version:", np.__version__)
print("NumPy matches 2.1.3:", np.__version__ == "2.1.3")
print("Python location:", sys.executable)
print("Working folder:", Path.cwd())
print("Rows and columns loaded:", commute.shape)

Python version: 3.13.15
Python 3.13: True
pandas version: 2.2.3
pandas matches 2.2.3: True
NumPy version: 2.1.3
NumPy matches 2.1.3: True
Python location: /Users/Angie/compss-211a/.venv/bin/python
Working folder: /Users/Angie/compss-211a/lab
Rows and columns loaded: (96, 9)


If an import fails or a version check says `False`, show the output to the instructor or GSI.

## Live-coding warm-up

Type **one line per cell** while the instructor demonstrates it. Before running each line, predict what it will show. The 50% value in `.describe()` is the median; the 75% value is the upper quartile. You will use these values to read the later program comparison.

Inspect `commute.columns`. Which column contains monthly transportation costs?

In [23]:
commute.columns

Index(['respondent_id', 'program', 'commute_mode', 'commute_minutes_one_way',
       'days_on_campus', 'leaves_after_8pm_days', 'monthly_transport_cost_usd',
       'wellbeing_score', 'reliable_internet'],
      dtype='object')

Select `program`, `commute_mode`, and `monthly_transport_cost_usd`, then show the first five rows.

In [24]:
commute[["program", "commute_mode", "monthly_transport_cost_usd"]].head()

,program,commute_mode,monthly_transport_cost_usd
0,Information,Walk,0.00
1,Public Policy,Walk,0.00
2,Public Policy,Bike,92.27
3,Social Welfare,Remote,0.00
4,Public Policy,Walk,0.00


Count missing values in `monthly_transport_cost_usd`.

In [6]:
commute[["monthly_transport_cost_usd"]].isna().sum()

monthly_transport_cost_usd    2
dtype: int64

Make `known_costs` from the cost column with `.dropna()`. Keep the original `commute` table unchanged.

In [25]:
known_costs = commute[["monthly_transport_cost_usd"]].dropna()

Run `.describe()` on `known_costs`. Find the 50% and 75% values in its output.

In [8]:
known_costs.describe()

,monthly_transport_cost_usd
count,94.000000
mean,64.899468
std,58.766299
min,0.000000
25%,0.000000
50%,63.420000
75%,115.862500
max,197.010000


## Duo Task 1 · Select campus commuters

To compare costs for getting to campus, keep respondents whose `commute_mode` is not `"Remote"`. Remote respondents have recorded zero campus days and zero transportation costs. Those zeros are valid recorded values, but this comparison focuses on campus commuters.

Create `commuters` with `.copy()`. Display:

- the number of rows and the number of missing costs;
- `.describe()` for the nonremote costs, especially 50% and 75%; and
- a few rows with `respondent_id`, `program`, `commute_mode`, and `monthly_transport_cost_usd`.

Keep missing costs as missing values. If your filter is stuck for two minutes, ask the teaching team to check it.

<details><summary>First hint</summary>Look at the `commute_mode` column and use `!= "Remote"` to keep other rows.</details>

<details><summary>Next hint</summary>Try `commute[commute["commute_mode"] != "Remote"].copy()`. Use `.isna().sum()` to count unknown costs.</details>

In [49]:
commuters = commute[commute["commute_mode"] != "Remote"].copy()

In [45]:
commuters[["monthly_transport_cost_usd"]].isna().sum()
commuters


,respondent_id,program,commute_mode,commute_minutes_one_way,days_on_campus,leaves_after_8pm_days,monthly_transport_cost_usd,wellbeing_score,reliable_internet
0,BMS-001,Information,Walk,18.0,5,2,0.00,6.8,Yes
1,BMS-002,Public Policy,Walk,13.0,4,2,0.00,6.2,Yes
2,BMS-003,Public Policy,Bike,16.0,5,1,92.27,7.5,Yes
4,BMS-005,Public Policy,Walk,25.0,3,0,0.00,7.3,Yes
5,BMS-006,Public Policy,BART + walk,49.0,2,1,65.44,6.2,Yes
...,...,...,...,...,...,...,...,...,...
89,BMS-090,Information,AC Transit,41.0,3,1,46.46,6.1,Yes
91,BMS-092,Social Welfare,AC Transit,44.0,4,1,172.62,6.5,Yes
92,BMS-093,MaCSS,Walk,12.0,2,1,0.00,7.4,Yes
94,BMS-095,MaCSS,AC Transit,51.0,3,2,82.66,7.8,Yes


## Duo Task 2 · Create cost bands with a function

Create `cost_band(cost)`. It must return:

- `"unknown"` when the cost is missing;
- one label for lower observed costs;
- one label for middle observed costs; and
- one label for higher observed costs.

Choose two cutoffs in your function. Test both boundaries and a missing value. Then use `.apply()` to create `commuters["cost_band"]` and display the category counts. Switch who types and who checks the output after this task.

<details><summary>First hint</summary>Decide what to return for a missing value before you compare the cost with a number.</details>

<details><summary>Next hint</summary>Start with `if pd.isna(cost): return "unknown"`, then use `if` and `return` for your two cutoffs. Ask the teaching team to check one boundary test if needed.</details>

In [52]:
commuters["monthly_transport_cost_usd"].describe()
#lower bound is below 50%
#middle bound is between 50-75%
#high is above 75%


count     85.000000
mean      71.771176
std       57.657883
min        0.000000
25%        0.000000
50%       67.500000
75%      118.190000
max      197.010000
Name: monthly_transport_cost_usd, dtype: float64

In [40]:
def cost_band(cost):
    if cost is None or np.isnan(cost):
        return "unknown"
    if cost < 50:
        return "lower"
    elif cost < 75:
        return "middle"
    else:
        return "high"

In [54]:
#analysis_posts['transit_keyword'] = analysis_posts['text'].apply(transit_keyword)

commuters["cost_band"] = commuters["monthly_transport_cost_usd"].apply(cost_band)

In [55]:
commuters.head()

,respondent_id,program,commute_mode,commute_minutes_one_way,days_on_campus,leaves_after_8pm_days,monthly_transport_cost_usd,wellbeing_score,reliable_internet,cost_band
0,BMS-001,Information,Walk,18.0,5,2,0.00,6.8,Yes,lower
1,BMS-002,Public Policy,Walk,13.0,4,2,0.00,6.2,Yes,lower
2,BMS-003,Public Policy,Bike,16.0,5,1,92.27,7.5,Yes,high
4,BMS-005,Public Policy,Walk,25.0,3,0,0.00,7.3,Yes,lower
5,BMS-006,Public Policy,BART + walk,49.0,2,1,65.44,6.2,Yes,middle


## Duo Task 3 · Compare academic programs

Group `commuters` by `program`. Create `program_summary` with:

- the number of respondents;
- the number of known transportation costs; and
- the median monthly transportation cost.

Sort the largest median to the top. Display the table and the 50% and 75% cost values for all commuters from Task 1. Check which program leads and whether any program has fewer known costs than respondents.

<details><summary>First hint</summary>Use `.groupby("program")`; median ignores missing cost values.</details>

<details><summary>Next hint</summary>Use `.agg(...)`. Count `respondent_id` for respondents and count the cost column for known costs. Sort by the median column with `ascending=False`.</details>

In [59]:
commuters_grouped = commuters.groupby("program")


In [ ]:
#platform_summary = topic_posts.groupby("platform")["likes"].agg(["size", "count", "mean", "median"])
#platform_summary


## Duo Task 4 · Change one analysis choice

Recompute the comparison after changing **one** choice. Choose one:

- use the mean instead of the median;
- include remote respondents;
- compare commute modes instead of programs; or
- use your cost bands.

Display the new table alongside `program_summary` and check whether the ordering changed. The mean option keeps the same programs and is a good place to start.

<details><summary>First hint</summary>Start from your Task 3 code and change just one part.</details>

<details><summary>Next hint</summary>For the mean option, replace `"median"` with `"mean"` in your group summary and give the new result a new name.</details>

## Exit

Save your notebook.